# 04 · DenseNet / Inception on CIFAR-10 —— 连接拓扑的两种答案

**家族位置**：`02_CNN_Family` 第 4 个项目（前：`03_ResNet_CIFAR10` ★ 已完成五臂消融）。

**与上一站的衔接**：ResNet 证明了"连接"是深度的解药——但"怎么连"不止一种答案。本章对比另外两种拓扑：**DenseNet**（每层接所有层，特征复用做到极致）与 **Inception**（多尺度并行分支，宽度换深度）。协议沿用 03 的最终配方（SGD-momentum-wd），并复用 ResNet20 作为**跨项目锚点**——03 的 61.63% 是同一协议下的数字，直接可比。

**学习目标**
1. DenseBlock 的通道拼接机制与通道数推演（growth / Transition 压缩）
2. Inception 块的四分支并行设计与 1×1 卷积的"降维再卷"省法
3. 三种拓扑（相加/拼接/并行）同协议同台
4. 理解 DenseNet 的工程痛点：通道数线性增长 → 显存与计算的实际约束（吞吐实测）

## 1. 原理：相加 vs 拼接 vs 并行

### 三种拓扑一句话

- **ResNet（相加）**：$y = F(x) + x$——维度必须对齐，特征是"累加"的，旧特征被覆盖式混合
- **DenseNet（拼接）**：$y = [x, F_1(x), F_2(x), \dots]$——所有历史特征原样保留，**任何一层都能直接访问任何早期特征**
- **Inception（并行）**：$y = [B_1(x), B_3(x), B_5(x), B_p(x)]$——同一输入同时过 1×1/3×3/5×5 四条分支，**让网络自己学该看多大感受野**

### DenseNet 的账：通道数怎么长

DenseBlock 每层新增 `growth` 个通道（本项目 growth=12），块内 6 层后通道 = 16 + 6×12 = 88；Transition 层用 1×1 卷积**压缩一半**（θ=0.5）再池化。三块下来最终 130 通道出 GAP——参数量 15.0 万（ResNet20 的 55%）。

**代价**：concat 让每层的输入通道数线性增长，计算/显存开销大——下面吞吐实测会看到 DenseNet 比 ResNet 慢 **4 倍**（118 vs 472 img/s），这是它工业落地少的真实原因之一。

### Inception 的账：四分支怎么省

朴素 5×5 卷积参数 = 25C²；Inception 先 1×1 降维再 5×5（c5 远小于 C），参数省一个数量级。"**降维再卷**"的 1×1 思想来自 NiN（02 项目），在 GoogLeNet 里成为标配。

In [ ]:
import sys
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import CIFAR10_CLASSES, load_cifar10_torch
from common.engine import fit
from common.models import DenseNetCIFAR, InceptionCIFAR, ResNetCIFAR
from common.utils import count_params, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

## 2. 数据：CIFAR-10（同 03 项目）

同一份数据、同一子集划分（前 10k）、同一标准化——保证与 03 的 ResNet 锚点严格可比。

In [ ]:
Xtr, ytr, Xte, yte = load_cifar10_torch(str(ROOT / "data"))
print("训练集:", Xtr.shape, "| 测试集:", Xte.shape)

fig, axes = plt.subplots(2, 10, figsize=(12, 3.0))
rng = np.random.default_rng(0)
for r in range(2):
    for c in range(10):
        idx = int(np.where(ytr.numpy() == c)[0][r])
        img = Xtr[idx].permute(1, 2, 0).numpy()
        img = (img * [0.2470, 0.2435, 0.2616] + [0.4914, 0.4822, 0.4465]).clip(0, 1)
        axes[r, c].imshow(img)
        axes[r, c].set_title(CIFAR10_CLASSES[c], fontsize=8)
        axes[r, c].axis("off")
plt.suptitle("CIFAR-10：每类两个样本", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 模型定稿与参数/通道推演

DenseNet 的通道推演在代码里逐步打印核对；Inception 打印每个块的分支通道数（拼接后总通道）。

In [ ]:
growth = 12
ch = 16
print("== DenseNet 通道推演（growth=12, θ=0.5）==")
print(f"stem 输出: {ch}")
for i in range(3):
    ch += 6 * growth
    print(f"DenseBlock{i+1} 后: {ch} (= 前值 + 6×12)")
    if i < 2:
        ch = int(ch * 0.5)
        print(f"Transition{i+1} 后: {ch} (压缩一半)")
print(f"最终 GAP 前: {ch} 通道")

m = DenseNetCIFAR()
print(f"\nDenseNetCIFAR 参数量: {count_params(m):,}")

m = InceptionCIFAR()
print(f"InceptionCIFAR 参数量: {count_params(m):,}（四分支: 1×1 / 3×3 / 5×5 / pool+1×1）")

ARMS = {
    "ResNet20(锚点)": lambda: ResNetCIFAR(),
    "DenseNetCIFAR": DenseNetCIFAR,
    "InceptionCIFAR": InceptionCIFAR,
}
for name, cls in ARMS.items():
    print(f"{name:16s} 参数量={count_params(cls()):>8,}")

## 4. 主实验：三拓扑同台（约 40 分钟 CPU）

**协议（沿用 03 最终配方）**：10k 训练子集、10 epochs、SGD(momentum=0.9, lr=0.05) + weight_decay=1e-4、batch 128、seed=0、测试全量 10k、无增强。ResNet20 在 03 项目同协议 15 epochs 的 61.63% 作为锚点参照。

预期：DenseNet 凭特征复用应超过同量级参数的 ResNet20；Inception 的多尺度应带来不错的起点效率。

In [ ]:
EPOCHS = 10
tr = DataLoader(TensorDataset(Xtr[:10000], ytr[:10000]), batch_size=128, shuffle=True)
te = DataLoader(TensorDataset(Xte, yte), batch_size=512)

results = {}
for name, cls in ARMS.items():
    set_seed(0)
    hist = fit(cls(), tr, te, epochs=EPOCHS, lr=0.05, device=DEVICE,
               optimizer_cls=partial(torch.optim.SGD, momentum=0.9),
               weight_decay=1e-4, verbose=False)
    results[name] = hist
    print(f"{name:16s} val_acc={hist['val_acc'][-1]:.2%} | val_loss={hist['val_loss'][-1]:.4f} | train_acc={hist['train_acc'][-1]:.2%}", flush=True)

print("\n锚点：ResNet20 @03项目同协议15ep = 61.63%")

In [ ]:
colors = {"ResNet20(锚点)": "#DD8452", "DenseNetCIFAR": "#4C72B0", "InceptionCIFAR": "#55A868"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for name, c in colors.items():
    axes[0].plot(results[name]["val_acc"], marker="o", ms=4, label=name, color=c)
    axes[1].plot(results[name]["train_acc"], marker="o", ms=4, label=name, color=c)
axes[0].set_title("val_acc（CIFAR-10 全量 10k 测试）")
axes[1].set_title("train_acc（10k 子集）")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

accs = {n: results[n]["val_acc"][-1] for n in ARMS}
fig, ax = plt.subplots(figsize=(7.5, 4))
names = list(ARMS)
bars = ax.bar(names, [accs[n] for n in names], color=[colors[n] for n in names])
for b, n in zip(bars, names):
    ax.text(b.get_x() + b.get_width() / 2, accs[n], f"{accs[n]:.2%}", ha="center", va="bottom", fontsize=9)
ax.axhline(0.6163, color="#DD8452", ls="--", lw=1, alpha=0.7)
ax.text(2.45, 0.6205, "ResNet20 @15ep", fontsize=8, color="#DD8452")
ax.set_ylim(0.3, 0.80); ax.set_ylabel("val_acc")
ax.set_title("三种连接拓扑同台（10k 子集 · 10 epochs · SGD-momentum-wd · seed=0）")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 效率视角：三种拓扑的"性价比"

精度只是半个故事——DenseNet 的拼接让计算图变宽（吞吐约为 ResNet 的 1/4），这是它工业落地少于 ResNet 的真实原因。画"精度-参数量"散点图，把拓扑选择放回工程语境。

In [ ]:
params = {n: count_params(cls()) for n, cls in ARMS.items()}
thr = {"ResNet20(锚点)": 474, "DenseNetCIFAR": 121, "InceptionCIFAR": 548}  # 吞吐冒烟实测 img/s

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for name, c in colors.items():
    axes[0].scatter(params[name] / 1e3, accs[name], s=180, color=c, zorder=3)
    axes[0].annotate(f"{name}\n{accs[name]:.2%} · {params[name]/1e3:.0f}k", (params[name] / 1e3, accs[name]),
                     textcoords="offset points", xytext=(8, -14), fontsize=8)
    axes[1].bar(name, 10000 / thr[name] * 2, color=c)  # 1 epoch 训练+评估耗时估算(×2 保守)
axes[0].set_xlabel("参数量 (千)"); axes[0].set_ylabel("val_acc"); axes[0].set_title("精度 vs 参数量")
axes[0].grid(alpha=0.3)
axes[1].set_ylabel("每 epoch 相对耗时（倍）"); axes[1].set_title("训练成本（吞吐实测换算，ResNet=1x 基准）")
axes[1].bar(0, 1.0, color=colors["ResNet20(锚点)"])
plt.tight_layout()
plt.savefig(FIGS / "fig3_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{'模型':16s} {'参数量':>9s} {'val_acc':>8s} {'吞吐(img/s)':>10s}")
for n in ARMS:
    print(f"{n:16s} {params[n]:>9,} {accs[n]:>8.2%} {thr[n]:>10d}")

## 6. 冠军的错误

冠军模型的混淆矩阵——看拓扑差异是否改变错误分布（03 的 ResNet32 是 dog→cat 231 次领跑）。

In [ ]:
best_name = max(accs, key=accs.get)
print("冠军:", best_name, f"{accs[best_name]:.2%}")
set_seed(0)
best_model = ARMS[best_name]()
fit(best_model, tr, te, epochs=EPOCHS, lr=0.05, device=DEVICE,
    optimizer_cls=partial(torch.optim.SGD, momentum=0.9), weight_decay=1e-4, verbose=False)
best_model.eval()
with torch.no_grad():
    pred = best_model(Xte).argmax(1)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte.numpy(), pred.numpy())
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(CIFAR10_CLASSES, fontsize=8)
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=6)
ax.set_xlabel("预测"); ax.set_ylabel("真实"); ax.set_title(f"{best_name} 混淆矩阵")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(FIGS / "fig4_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
flat = cm_off.ravel().argsort()[::-1][:5]
for k in flat:
    r, c = np.unravel_index(k, cm.shape)
    print(f"  {CIFAR10_CLASSES[r]:10s} → {CIFAR10_CLASSES[c]:10s} : {cm_off[r, c]} 次")

## 7. 总结与下一步

**本项目收获**

1. 三种拓扑的机制与账目：相加（对齐约束）/ 拼接（通道线性增长+Transition 压缩）/ 并行（1×1 降维再卷）
2. 同协议同台的精度对比（结果见 §4）+ 效率视角（DenseNet 慢 4 倍的工程代价）
3. 跨项目锚点方法论：同协议数字直接可比，家族实验形成累积证据链

**下一步**：`05_Lightweight_MobileNet_EfficientNet`——效率时代：深度可分离卷积（MobileNet）与复合缩放（EfficientNet），画 Acc-FLOPs 权衡曲线，给本家族的"效率视角"收口。